# 4 · Transfer learning: event classifiers from base-classifier outputs

Retrain an **event classifier** for new labels (species, populations, call types...) on the outputs of classifiers you already have, without retraining those base classifiers.

**Base classifiers** can be anything that writes predictions in the [standard classifier output format](../../schemas/classifier_output.md): delphinID models run in PAMGuard (read straight from the PAMGuard database), random forests from notebook 3, or others, in any combination.

**How it works**
1. Base-classifier predictions are grouped into **events** (labelled time periods, recordings, or fixed time bins).
2. Each event is summarised by the **mean probability of each class from each base classifier**, e.g. `delphinID_whistles|Dde`. A base classifier with no predictions in an event contributes zeros.
3. A random forest learns to map these event features to your labels. It's tested by leaving out one group (e.g. encounter) at a time.

**Outputs** (in `OUTPUT_DIR`): event predictions from cross-validation (standard format), accuracy summary, the model (`.joblib`), and predictions for new data.

By default it runs on a small **synthetic** example (`examples/base_predictions_example_synthetic.csv`, `examples/events_example_synthetic.csv`).

## Setup
Locally: `pip install -e "python[notebooks]"` from the repository folder. In Google Colab, run the next cell.

In [ ]:
# Google Colab only
# %pip install -q "speciesclassifiers @ git+https://github.com/tristankleyn/speciesClassifiers#subdirectory=python"

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from speciesclassifiers.io import classes_from_model, list_prediction_tables, read_dl_predictions, read_recordings
from speciesclassifiers.plots import plot_confusion, plot_importance
from speciesclassifiers.randomforest import RandomForestModel
from speciesclassifiers.transferlearning import (assign_events, cross_validate, event_features, event_params,
                                                 filter_events, fit, predict, summarise, threshold_curve,
                                                 time_bin_events)

## 1 · Load base-classifier predictions
Choose **one** source:
- `"csv"`: one or more CSVs in the standard classifier output format (needs a `time` column).
- `"database"`: deep learning prediction tables in a PAMGuard database. PAMGuard doesn't store class names, so give them for each table, either as a list or as the path to the model zip (its `.pdtf` lists them).

In [ ]:
SOURCE = "csv"   # "csv" or "database"

# csv
PREDICTION_CSVS = ["../../examples/base_predictions_example_synthetic.csv"]

# database
DATABASE = "path/to/database.sqlite3"
TABLES = {   # table name -> (name to use, vocalisation type, class names or model zip)
    "Deep_Learning_Classifier___Whistles": ("delphinID_whistles", "whistle", ["Dde", "Ggr", "Gme", "Lac", "Lal", "Oor", "Ttr"]),
    "Deep_Learning_Classifier___Clicks":   ("delphinID_clicks",   "click",   ["Dde", "Ggr", "Gme", "Lal", "Ttr"]),
}
OUTPUT_DIR = "transferlearning_run"

out = Path(OUTPUT_DIR); out.mkdir(parents=True, exist_ok=True)
if SOURCE == "csv":
    preds = pd.concat([pd.read_csv(f) for f in PREDICTION_CSVS], ignore_index=True)
else:
    print("Prediction tables in the database:", list_prediction_tables(DATABASE))
    parts = []
    for table, (name, voc, classes) in TABLES.items():
        classes = classes_from_model(classes) if isinstance(classes, str) else classes
        parts.append(read_dl_predictions(DATABASE, table, classes, name, voc))
    preds = pd.concat(parts, ignore_index=True)
preds.groupby("classifier").agg(predictions=("detection_id", "nunique"), classes=("class", "nunique"))

## 2 · Define events
- `"csv"`: an events CSV in the annotations format: `event_id, label, start, end` (UTC), plus a grouping column such as `encounter` and any others. **Needed for training.**
- `"recordings"`: one event per recording file in the database's Sound_Acquisition table.
- `"bins"`: fixed time bins of `BIN_MINUTES`.

To label events made from recordings or bins, as in Classify-delphinID, run with `"recordings"` or `"bins"` and `EXPORT_FOR_LABELLING = True`. Then open `events_to_label.csv`, fill in `label` (and e.g. `encounter`), and come back with `EVENTS = "csv"` pointing to it.

In [ ]:
EVENTS = "csv"            # "csv", "recordings" or "bins"
EVENTS_CSV = "../../examples/events_example_synthetic.csv"
BIN_MINUTES = 10
EXPORT_FOR_LABELLING = False

if EVENTS == "csv":
    assigned = assign_events(preds, EVENTS_CSV)
elif EVENTS == "recordings":
    rec = read_recordings(DATABASE)
    assigned = assign_events(preds, rec)
else:
    assigned = time_bin_events(preds, BIN_MINUTES)
events, features = event_features(assigned)

if EXPORT_FOR_LABELLING:
    t = events[["event_id", "first_detection", "last_detection"]].copy()
    if EVENTS == "recordings":
        t = rec.merge(t[["event_id"]], on="event_id")
    else:
        t = t.rename(columns={"first_detection": "start", "last_detection": "end"})
        if EVENTS == "bins":
            t["start"] = pd.to_datetime(t["event_id"]).astype("int64") // 10**9
            t["end"] = t["start"] + BIN_MINUTES * 60
    for c in ["start", "end"]:
        t[c] = pd.to_datetime(t[c], unit="s", utc=True).dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]
    t["label"], t["encounter"] = "", ""
    t.to_csv(out / "events_to_label.csv", index=False)
    print(f"Wrote {len(t)} events to {out / 'events_to_label.csv'}: add labels, then use EVENTS = 'csv'")

print(f"{len(events)} events, {len(features)} features")
events.head()

## 3 · Filter events
Events with few base-classifier predictions carry little information. `MIN_DETECTIONS` sets a minimum per base classifier; `MODE = "or"` keeps events meeting any minimum, `"and"` only those meeting all.

In [ ]:
LABEL_COL = "label"
GROUP_COL = "encounter"     # independent unit for testing; "event_id" = each event on its own
MIN_DETECTIONS = {"delphinID_clicks": 5, "delphinID_whistles": 3}
MODE = "or"

events_f = filter_events(events, MIN_DETECTIONS, MODE)
events_f = events_f[events_f[LABEL_COL].notna() & (events_f[LABEL_COL].astype(str) != "")]
events_f[LABEL_COL] = events_f[LABEL_COL].astype(str)
print(f"{len(events_f)} of {len(events)} events kept")
events_f.groupby(LABEL_COL).agg(groups=(GROUP_COL, "nunique"), events=(GROUP_COL, "size"))

Events in the space of their features (first two principal components), coloured by label. Labels that separate here are easier to classify.

In [ ]:
x = events_f[features].to_numpy(float)
x = x - x.mean(0)
pcs = x @ np.linalg.svd(x, full_matrices=False)[2][:2].T
fig, ax = plt.subplots(figsize=(6, 4.5))
for lab in sorted(events_f[LABEL_COL].unique()):
    m = (events_f[LABEL_COL] == lab).to_numpy()
    ax.scatter(pcs[m, 0], pcs[m, 1], s=20, alpha=0.7, label=lab)
ax.set(xlabel="PC1", ylabel="PC2"); ax.legend(); plt.show()

## 4 · Train and test

In [ ]:
N_TREES   = 1000
MTRY      = None   # features tried per split; None = square root of the number of features
NODE_SIZE = 5      # minimum events per leaf
PRUNE     = 0.0    # share of each label's training events to drop as outliers
SEED      = 42

params = event_params(N_TREES, MTRY, NODE_SIZE, None, PRUNE, SEED)
cv = cross_validate(events_f, features, LABEL_COL, GROUP_COL, params)
cv.to_csv(out / "cv_event_predictions.csv", index=False)

## Results
`MIN_SCORE` discards events whose decision score (p1 × (p1 − p2)) is below it: fewer events classified, more of them correct.

In [ ]:
MIN_SCORE = 0.0
s = summarise(cv, MIN_SCORE)
print(f"Event accuracy: {s['event_accuracy']:.1%}   Mean class accuracy: {s['mean_class_accuracy']:.1%}   "
      f"Events discarded: {s['events_discarded']:.1%}")
display(s["accuracy_by_class"].rename("accuracy").to_frame().style.format("{:.0%}"))
plot_confusion(s["confusion"], "Events"); plt.show()

In [ ]:
ev = s["events"].assign(correct=lambda d: d["predicted"] == d["true_class"])
ev[["event", "group", "true_class", "predicted", "correct", "score"]].sort_values(["correct", "score"])

In [ ]:
curve = threshold_curve(cv)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(curve["threshold"], curve["event_accuracy"], label="event accuracy")
ax.plot(curve["threshold"], curve["mean_class_accuracy"], label="mean class accuracy")
ax.plot(curve["threshold"], curve["events_kept"], "--", label="events kept")
ax.set(xlabel="Minimum score", ylim=(0, 1.05)); ax.legend(); plt.show()

## 5 · Final model
Trained on all filtered events. Feature importance shows which base-classifier outputs the new classifier relies on.

In [ ]:
model = fit(events_f, features, LABEL_COL, GROUP_COL, params)
model.save(out / "event_classifier.joblib")
(out / "summary.json").write_text(json.dumps({
    "params": params.to_dict(), "features": features, "classes": model.classes, "min_detections": MIN_DETECTIONS,
    "mode": MODE, "event_accuracy": s["event_accuracy"], "mean_class_accuracy": s["mean_class_accuracy"]}, indent=1))
plot_importance(model.importance()); plt.tight_layout(); plt.show()

## 6 · Classify new data
Load new base-classifier predictions (same base classifiers and names as for training), make events, and apply the model. Use the same `MIN_DETECTIONS` filter as in training.

In [ ]:
RUN_NEW = False
NEW_PREDICTION_CSVS = ["path/to/new_predictions.csv"]   # or build `new_preds` from a database as in step 1
NEW_EVENTS = "bins"          # "csv", "recordings" (needs NEW_DATABASE) or "bins"
NEW_EVENTS_CSV = "path/to/new_events.csv"
NEW_DATABASE = "path/to/new_database.sqlite3"
MODEL_PATH = out / "event_classifier.joblib"

if RUN_NEW:
    m = RandomForestModel.load(MODEL_PATH)
    new_preds = pd.concat([pd.read_csv(f) for f in NEW_PREDICTION_CSVS], ignore_index=True)
    if NEW_EVENTS == "csv":
        a = assign_events(new_preds, NEW_EVENTS_CSV)
    elif NEW_EVENTS == "recordings":
        a = assign_events(new_preds, read_recordings(NEW_DATABASE))
    else:
        a = time_bin_events(new_preds, BIN_MINUTES)
    new_events, _ = event_features(a)
    new_events = filter_events(new_events, MIN_DETECTIONS, MODE)
    result = predict(m, new_events)
    result.to_csv(out / "new_event_predictions.csv", index=False)
    display(result[["event_id", "predicted", "score", *m.classes]])